<a href="https://colab.research.google.com/github/algalda/MATH-344L-project/blob/main/02_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists('ml-100k'):
    urllib.request.urlretrieve(
        'https://files.grouplens.org/datasets/movielens/ml-100k.zip',
        'ml-100k.zip')
    zipfile.ZipFile('ml-100k.zip').extractall()

ratings = pd.read_csv(
    'ml-100k/u.data', sep='\t',
    names=['user', 'item', 'rating', 'time'])

print(ratings.head())
print('Number of ratings:', len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


- downloads and unzips ml-100k.
- reads ratings into table

In [12]:
def split_ratings(ratings, test_frac=0.10, seed=0):
    """Hide ratings without removing a user's or movie's last training rating."""
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby('user').size().to_dict()
    item_left = ratings.groupby('item').size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row['user'], row['item']

        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print('train:', len(train), 'test:', len(test))
print('test fraction:', round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


- hides some ratings to create test set
- keeps min. 1 rating per user & movie
- uses seed to have recreatable split

In [13]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df['user'].to_numpy() - 1,
      df['item'].to_numpy() - 1] = df['rating'].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)

print('Matrix shape:', R.shape)
print('Known training ratings:', known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


- builds matrix from training ratings
- fills in missing ratings as 'np.nan'

In [14]:
truth = test['rating'].to_numpy()
u = test['user'].to_numpy() - 1
i = test['item'].to_numpy() - 1

- stores real test ratings in 'truth'
- set u equal to each test ratings user-1
- set i equal to each test ratings movie id-1

In [15]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

- calculates rmse
- gives result in stars where lower is better


In [16]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

print('Overall average:', round(mu, 3))

Overall average: 3.53


- calculates mu
- calculates each users average
- calculates each movies average

In [17]:
scores = {}
scores['always 3'] = rmse(np.full(len(test), 3.0), truth)
scores['overall average'] = rmse(np.full(len(test), mu), truth)
scores['user average'] = rmse(user_mean[u], truth)
scores['movie average'] = rmse(item_mean[i], truth)

for name, score in scores.items():
    print(f'{name:16s} {score:.4f}')

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


- scores 4 predictors on test ratings
- prints scores


In [18]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)

    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]

    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0
    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])

    return mu, b_user, b_item

- calculates mu + user & movie effect


In [19]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

scores['user + movie effects'] = rmse(pred, truth)
print(pd.Series(scores, name='test RMSE').round(4).sort_values())

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


- predicts mu + user effect + movie effect
- clips predictions between 1 and 5
- adds score to other baselines, sorts, prints